# Practical 2B: Binary Classification using DNN on IMDB Movie Reviews

## Aim
Classify movie reviews into positive and negative reviews using a Deep Neural Network.

## Dataset
Local file used in this notebook: dl/2/IMDB Dataset.csv

## What you will learn
- How to load text data
- How to clean basic HTML tags from reviews
- How to convert text into numbers using TextVectorization
- How to build a simple DNN text classifier
- How binary classification uses sigmoid output and binary crossentropy

Note: This notebook uses TensorFlow/Keras. No PyTorch code is used.

In [ ]:
# If imports fail on your machine, install these packages once:
# !pip install pandas numpy scikit-learn matplotlib tensorflow

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Step 1: Load the dataset

The dataset has two columns: review and sentiment.

In [ ]:
DATA_PATH = Path("dl/2/IMDB Dataset.csv")

data = pd.read_csv(DATA_PATH)
print("Dataset shape:", data.shape)
data.head()

In [ ]:
print(data.info())
print("\nSentiment count:")
print(data["sentiment"].value_counts())

## Step 2: Clean text and encode labels

We remove HTML break tags and extra spaces. Then positive becomes 1 and negative becomes 0.

In [ ]:
def clean_review(text):
    text = re.sub(r"<br\s*/?>", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


data["review"] = data["review"].apply(clean_review)
data["label"] = data["sentiment"].map({"negative": 0, "positive": 1})

data[["review", "sentiment", "label"]].head()

## Step 3: Split train and test data

We keep the label distribution balanced using stratify.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["review"].values,
    data["label"].values,
    test_size=0.2,
    random_state=SEED,
    stratify=data["label"].values
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

## Step 4: Convert text to numbers

TextVectorization builds a vocabulary from the training reviews. It converts every review into a fixed-length sequence of word indexes.

In [ ]:
MAX_TOKENS = 10000
SEQUENCE_LENGTH = 200

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH
)

vectorizer.adapt(X_train)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("First 10 words:", vectorizer.get_vocabulary()[:10])

## Step 5: Build the binary text classification model

Embedding learns a small numeric vector for every word. GlobalAveragePooling1D converts the full review into one vector. The final sigmoid neuron gives the probability of a positive review.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    layers.Embedding(input_dim=MAX_TOKENS, output_dim=64),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 6: Train the model

Three epochs are enough for a classroom practical. Increase epochs if you want slightly better accuracy.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=128,
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("IMDB Sentiment DNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Step 7: Evaluate the model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")

probabilities = model.predict(X_test, verbose=0).flatten()
y_pred = (probabilities >= 0.5).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=["negative", "positive"]
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["negative", "positive"]
)
display.plot(cmap="Blues", colorbar=False)
plt.title("IMDB Sentiment Confusion Matrix")
plt.show()

## Step 8: Predict custom reviews

In [ ]:
sample_reviews = np.array([
    "The movie was wonderful and the acting was excellent.",
    "This film was boring, slow, and a waste of time.",
    "The story was simple but the performances were enjoyable."
])

sample_probabilities = model.predict(sample_reviews, verbose=0).flatten()

for review, probability in zip(sample_reviews, sample_probabilities):
    sentiment = "positive" if probability >= 0.5 else "negative"
    print(f"Review: {review}")
    print(f"Predicted sentiment: {sentiment} ({probability:.3f})")
    print()

## Conclusion

We created a binary DNN classifier for IMDB reviews. Binary classification uses one sigmoid output neuron and binary crossentropy loss.